In [ ]:
# 本文件主要进行数据清洗和股票池筛选，具体标准见书P103
%run config_0.ipynb
%run data_io_1.ipynb

import pandas as pd
import numpy as np

In [ ]:
# 所有表共用的字段契约与类型规范化函数
def require_columns(frame, columns, table_name):
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise KeyError('{}缺少字段: {}'.format(table_name, missing))


def normalize_stock_code(series):
    """把 1、1.0、'000001' 等股票代码统一成六位字符串。"""
    numeric = pd.to_numeric(series.astype('string').str.strip(), errors='coerce')
    whole_number = numeric.where(numeric.mod(1).eq(0))
    in_a_share_range = whole_number.between(0, 999999)
    return (
        whole_number.where(in_a_share_range)
        .astype('Int64')
        .astype('string')
        .str.zfill(6)
    )


def normalize_datetime(series):
    return pd.to_datetime(series, errors='coerce')


def normalize_numeric(series):
    return pd.to_numeric(series, errors='coerce')


In [ ]:
# 日股数据

# 进行数据清洗
def cleaning(df):
    required = [
        stock_code, date, close_price, total_shares, daily_ret,
        daily_rfret, pb, roe, trade_volumn, trade_amount,
    ]
    require_columns(df, required, '日股表')

    df[stock_code] = normalize_stock_code(df[stock_code])
    df[date] = normalize_datetime(df[date])
    for col in [
        close_price, total_shares, daily_ret, daily_rfret,
        pb, roe, trade_volumn, trade_amount,
    ]:
        df[col] = normalize_numeric(df[col])

    df = df.dropna(subset=required)
    df = df.drop_duplicates(subset=[stock_code, date])

    for col in [
        close_price, trade_amount, trade_volumn,
        total_shares, daily_ret, daily_rfret,
    ]:
        lower = df[col].quantile(0.01)
        upper = df[col].quantile(0.99)
        df[col] = df[col].clip(lower, upper)

    return df


In [ ]:
# 日股数据

# 股票池筛选
def universe(df):
    # 黑名单处理：只保留正常上市状态(Norm)的股票
    df = df[df[listed_status] == 'Norm']

    # 全样本交易日历（排序后的所有交易日）
    calendar = np.sort(df[date].unique())

    # 每只股票的首末交易日
    first = df.groupby(stock_code)[date].transform('min')
    last = df.groupby(stock_code)[date].transform('max')

    # 该股票自身的可交易时长 = 交易日历中落在 [首日, 末日] 之间的天数
    lo = np.searchsorted(calendar, first.values)
    hi = np.searchsorted(calendar, last.values, side='right')
    possible_days = pd.Series(hi - lo, index=df.index)

    # 该股票实际交易天数
    actual_days = df.groupby(stock_code)[date].transform('nunique')

    # 最少交易日：实际交易天数 >= 自身可交易时长的 2/3
    df = df[actual_days >= (2 / 3) * possible_days]
    return df


In [ ]:
# 资产负债表数据

# 进行数据清洗
def bs_cleaning(bs_df):
    required = [bs_stock_code, bs_end_date, total_ass]
    require_columns(bs_df, required, '资产负债表')
    extra_required = [
        bs_public_date, total_cur_assets, total_cur_lia, short_loan,
        long_loan, tax_pay, cash_eqv, trad_ass, total_liab,
    ]
    require_columns(bs_df, extra_required, '资产负债表')

    bs_df[bs_stock_code] = normalize_stock_code(bs_df[bs_stock_code])
    bs_df[bs_end_date] = normalize_datetime(bs_df[bs_end_date])
    bs_df[bs_public_date] = normalize_datetime(bs_df[bs_public_date])
    for col in [
        total_ass, total_cur_assets, total_cur_lia, short_loan, long_loan,
        tax_pay, cash_eqv, trad_ass, total_liab,
    ]:
        bs_df[col] = normalize_numeric(bs_df[col])

    bs_df = bs_df.dropna(subset=required)
    bs_df = bs_df.drop_duplicates()
    return bs_df

In [ ]:
# 资产负债表数据

# 股票池筛选

In [ ]:
# 现金流量表数据

# 进行数据清洗
def cf_cleaning(cf_df):
    required = [
        cf_code, cf_report_type, cf_end_date,
        cf_public_date, cf_net_increase,
    ]
    require_columns(cf_df, required, '现金流量表')

    cf_df[cf_code] = normalize_stock_code(cf_df[cf_code])
    cf_df[cf_report_type] = (
        cf_df[cf_report_type].astype('string').str.strip().str.upper()
    )
    cf_df[cf_end_date] = normalize_datetime(cf_df[cf_end_date])
    cf_df[cf_public_date] = normalize_datetime(cf_df[cf_public_date])
    cf_df[cf_net_increase] = normalize_numeric(cf_df[cf_net_increase])
    cf_df[cf_intangible_amort] = normalize_numeric(cf_df[cf_intangible_amort])
    cf_df[cf_defer_amort] = normalize_numeric(cf_df[cf_defer_amort])
    
    cf_df = cf_df.dropna(subset=required)
    cf_df = cf_df.drop_duplicates()
    return cf_df

In [ ]:
# 固定资产及折旧表数据

# 进行数据清洗
def fas_cleaning(fas_df):
    required = [
        fas_code, fas_end_date, fas_report_type,
        fasdepr_type, fas_depr_incr,
    ]
    require_columns(fas_df, required, '固定资产及折旧表')

    fas_df[fas_code] = normalize_stock_code(fas_df[fas_code])
    fas_df[fas_report_type] = (
        fas_df[fas_report_type].astype('string').str.strip().str.upper()
    )
    fas_df[fasdepr_type] = fas_df[fasdepr_type].astype('string').str.strip()
    fas_df[fas_end_date] = normalize_datetime(fas_df[fas_end_date])
    fas_df[fas_depr_incr] = normalize_numeric(fas_df[fas_depr_incr])

    fas_df = fas_df.dropna(subset=required)
    fas_df = fas_df.drop_duplicates()
    return fas_df

In [ ]:
# 

In [ ]:
# 年度事件表构造与对齐（资产负债表 + 现金流量表 + 固定资产及折旧表公用）
# prepare 把已清洗的报表折叠成 (annual_stock_code, annual_available_date, annual_value) 三列事件表，
# align_annual_value 再把事件表向后对齐到日度主表；它们是 get_cp / get_asset_growth 的前置处理。
# 逐行语义详见 learning/特征构造模型/get_cp对齐防御逐行详解.md 与复杂因子构造详解.md。

def prepare_annual_cash_flow(cf_df):
    required = [
        cf_code, cf_report_type, cf_end_date,
        cf_public_date, cf_net_increase,
    ]
    require_columns(cf_df, required, '现金流量表')

    annual = cf_df[required].copy()
    # 记录源行号：后面的排序与 drop_duplicates(keep='last') 靠它让同键冲突的选择可复现。
    annual[annual_source_order] = np.arange(len(annual))
    # 只保留 Q4（年报）口径；现金流价格比的分子取最近一份年报，而非累计到任意季度。
    # 详见 learning/特征构造模型/get_cp对齐防御逐行详解.md 与 step4_merge_asof对齐.md。
    annual = annual.loc[annual[cf_report_type].eq('Q4')]

    # 排序把“物理行序”翻译成“经济时间序”，令下面 keep='last' 的“最后一行”有确定含义。
    # 详见 learning/基础函数使用手册/sort_values详解.md。
    annual = annual.sort_values([
        cf_code, cf_public_date, cf_end_date, annual_source_order,
    ])
    # 两级去重把主键唯一性落实成事实：先 (股票, 公告日, 报告期)，再 (股票, 公告日)，
    # 同一公告日同报告期冲突时保留源行号最大（最后写入）的那一份。
    # 详见 learning/基础函数使用手册/drop_duplicates详解.md。
    annual = annual.drop_duplicates(
        [cf_code, cf_public_date, cf_end_date], keep='last'
    )
    annual = annual.drop_duplicates(
        [cf_code, cf_public_date], keep='last'
    )

    # 再次排序后用 cummax 只保留报告期不倒退的事件（同一股票内报告期单调递增）。
    annual = annual.sort_values([cf_code, cf_public_date, cf_end_date])
    latest_report_end = annual.groupby(cf_code)[cf_end_date].cummax()
    annual = annual.loc[annual[cf_end_date].eq(latest_report_end)]

    annual = annual[[cf_code, cf_public_date, cf_net_increase]].rename(columns={
        cf_code: annual_stock_code,
        cf_public_date: annual_available_date,
        cf_net_increase: annual_value,
    })
    return annual.sort_values([
        annual_available_date, annual_stock_code,
    ]).reset_index(drop=True)


def prepare_annual_asset_growth(bs_df):
    required = [bs_stock_code, bs_end_date, total_ass]
    require_columns(bs_df, required, '资产负债表')

    annual = bs_df[required].copy()
    annual[annual_source_order] = np.arange(len(annual))
    annual = annual.loc[annual[bs_end_date].dt.month.eq(12)]
    annual = annual.sort_values([
        bs_stock_code, bs_end_date, annual_source_order,
    ])
    annual = annual.drop_duplicates(
        [bs_stock_code, bs_end_date], keep='last'
    )

    annual = annual.sort_values([bs_stock_code, bs_end_date])
    previous_assets = annual.groupby(bs_stock_code)[total_ass].shift(1)
    annual[annual_value] = (
        annual[total_ass] / previous_assets.where(previous_assets > 0)
    )
    annual = annual.loc[np.isfinite(annual[annual_value])]
    annual[annual_available_date] = (
        annual[bs_end_date] + pd.DateOffset(months=4)
    )
    annual = annual.rename(columns={
        bs_stock_code: annual_stock_code,
    })
    return annual[[
        annual_stock_code,
        annual_available_date,
        annual_value,
    ]].sort_values([
        annual_available_date, annual_stock_code,
    ]).reset_index(drop=True)

def prepare_annual_cf_amort(cf_df):
    required = [cf_code, cf_report_type, cf_end_date, cf_public_date,
                cf_intangible_amort, cf_defer_amort]
    require_columns(cf_df, required, '现金流量表')

    annual = cf_df[required].copy()
    annual[annual_source_order] = np.arange(len(annual))
    annual = annual.loc[annual[cf_report_type].eq('Q4')]
    # 折旧摊销公式里"摊销" = 两类摊销之和；列内缺失按已定口径当 0。
    annual[annual_value] = (
        annual[cf_intangible_amort].fillna(0) + annual[cf_defer_amort].fillna(0)
    )
    # 与 prepare_annual_cash_flow 同款：排序 → 两级去重 → cummax 防报告期倒退
    annual = annual.sort_values([cf_code, cf_public_date, cf_end_date, annual_source_order])
    annual = annual.drop_duplicates([cf_code, cf_public_date, cf_end_date], keep='last')
    annual = annual.drop_duplicates([cf_code, cf_public_date], keep='last')
    annual = annual.sort_values([cf_code, cf_public_date, cf_end_date])
    latest_report_end = annual.groupby(cf_code)[cf_end_date].cummax()
    annual = annual.loc[annual[cf_end_date].eq(latest_report_end)]

    annual = annual[[cf_code, cf_public_date, annual_value]].rename(columns={
        cf_code: annual_stock_code, cf_public_date: annual_available_date,
    })
    return annual.sort_values([annual_available_date, annual_stock_code]).reset_index(drop=True)


def prepare_annual_fixed_asset_depr(fas_df):
    required = [fas_code, fas_end_date, fasdepr_type, fas_report_type, fas_depr_incr]
    require_columns(fas_df, required, '固定资产及折旧表')

    annual = fas_df[required].copy()
    annual[annual_source_order] = np.arange(len(annual))
    annual = annual.loc[annual[fasdepr_type].eq('99999')]
    annual = annual.loc[annual[fas_report_type].eq('Q4')]
    annual = annual.sort_values([fas_code, fas_end_date, annual_source_order])
    annual = annual.drop_duplicates([fas_code, fas_end_date], keep='last')
    annual[annual_available_date] = annual[fas_end_date] + pd.DateOffset(months=4)

    annual = annual.rename(columns={fas_code: annual_stock_code, fas_depr_incr: annual_value})
    return annual[[annual_stock_code, annual_available_date, annual_value]].sort_values(
        [annual_available_date, annual_stock_code]
    ).reset_index(drop=True)


def align_annual_value(df, annual):
    require_columns(df, [stock_code, date], '日股表')
    require_columns(
        annual,
        [annual_stock_code, annual_available_date, annual_value],
        '年度事件表',
    )

    # 预填与 df 等长的全 NaN 数组，之后只回写匹配到的位置，保证行数、顺序、index 不变。
    # 空 df 或空事件表直接返回全缺失，不抛异常。
    # 详见 learning/基础函数使用手册/表粒度_主键与对齐详解.md。
    aligned = np.full(len(df), np.nan, dtype='float64')
    if df.empty or annual.empty:
        return aligned

    # 只取 (股票, 交易日) 键，并给每行一个“位置身份证” daily_row_position，供排序后按位置还原。
    # 详见 learning/基础函数使用手册/key保序详解.md。
    daily_keys = df[[stock_code, date]].copy().reset_index(drop=True)
    daily_keys = daily_keys.rename(columns={
        stock_code: annual_stock_code,
        date: daily_trade_date,
    })
    daily_keys[daily_row_position] = np.arange(len(daily_keys))
    valid_daily = daily_keys.dropna(
        subset=[annual_stock_code, daily_trade_date]
    )
    if valid_daily.empty:
        return aligned

    # merge_asof 做 point-in-time 对齐：同股票内取“可用日不晚于交易日”的最近一个年度事件；
    # direction='backward' 禁止未来信息，allow_exact_matches=True 允许公告日当天即可用。
    # 详见 learning/特征构造模型/step4_merge_asof对齐.md 与 learning/基础函数使用手册/merge_asof详解.md。
    matched = pd.merge_asof(
        valid_daily.sort_values([
            daily_trade_date, annual_stock_code,
        ]),
        annual.sort_values([
            annual_available_date, annual_stock_code,
        ]),
        left_on=daily_trade_date,
        right_on=annual_available_date,
        by=annual_stock_code,
        direction='backward',
        allow_exact_matches=True,
    )
    # 用匹配行携带的位置号把年度值写回定长数组，从而还原原 df 的行序与 index。
    positions = matched[daily_row_position].to_numpy(dtype='int64')
    aligned[positions] = matched[annual_value].to_numpy(dtype='float64')
    return aligned

In [ ]:
# 年度事件表构造（应计利润 + 净经营资产）
# prepare_annual_accruals：把 BS/CF/固定资产三份年报折叠成「年度应计利润」事件表
#   Accrual = (ΔCA − ΔCash) − (ΔCL − ΔSTD − ΔTP) − Dep
# prepare_annual_noa：把 BS 年报折叠成「年度净经营资产」事件表
#   NOA_t = (OA_t − OL_t) / TotalAssets_{t-1}
# 跨年差分只在年度宽面板上进行；口径与验证详见
# learning/特征构造模型/应计利润与净经营资产构造详解.md。

def prepare_annual_accruals(bs_df, cf_df, fas_df):
    """折叠三张报表为「年度应计利润」事件表。
    Accrual = (ΔCA − ΔCash) − (ΔCL − ΔSTD − ΔTP) − Dep
    跨年差分只在年度宽面板上进行；可用日取三类年报公告日中最晚者。
    """
    bs_required = [
        bs_stock_code, bs_end_date, bs_public_date,
        total_cur_assets, total_cur_lia, short_loan, tax_pay,
    ]
    cf_required = [
        cf_code, cf_report_type, cf_end_date, cf_public_date,
        cf_net_increase, cf_intangible_amort, cf_defer_amort,
    ]
    fas_required = [
        fas_code, fas_end_date, fas_report_type,
        fasdepr_type, fas_depr_incr,
    ]
    require_columns(bs_df, bs_required, '资产负债表')
    require_columns(cf_df, cf_required, '现金流量表')
    require_columns(fas_df, fas_required, '固定资产及折旧表')

    # 对于资产负债表年报面板：只留十二月年报，同键取最晚披露版本
    bs = bs_df[bs_required].copy()
    bs[annual_source_order] = np.arange(len(bs))
    bs = bs.loc[bs[bs_end_date].dt.month.eq(12)]
    bs = bs.sort_values([
        bs_stock_code, bs_end_date, bs_public_date, annual_source_order,
    ])
    bs = bs.drop_duplicates([bs_stock_code, bs_end_date], keep='last')
    bs = bs.rename(columns={
        bs_stock_code: annual_stock_code, bs_public_date: 'bs_pub',
    })
    bs['report_year'] = bs[bs_end_date].dt.year
    bs = bs.drop(columns=annual_source_order)

    # 对于现金流量表年报面板：Q4 + 两级去重 + 防报告期倒退；再取最晚披露版本
    cf = cf_df[cf_required].copy()
    cf[annual_source_order] = np.arange(len(cf))
    cf = cf.loc[cf[cf_report_type].eq('Q4')]
    cf = cf.sort_values([
        cf_code, cf_public_date, cf_end_date, annual_source_order,
    ])
    cf = cf.drop_duplicates([cf_code, cf_public_date, cf_end_date], keep='last')
    cf = cf.drop_duplicates([cf_code, cf_public_date], keep='last')
    cf = cf.sort_values([cf_code, cf_public_date, cf_end_date])
    latest_report_end = cf.groupby(cf_code)[cf_end_date].cummax()
    cf = cf.loc[cf[cf_end_date].eq(latest_report_end)]
    cf = cf.sort_values([cf_code, cf_end_date, cf_public_date])
    cf = cf.drop_duplicates([cf_code, cf_end_date], keep='last')
    cf = cf.rename(columns={
        cf_code: annual_stock_code, cf_public_date: 'cf_pub',
    })
    cf['report_year'] = cf[cf_end_date].dt.year
    cf = cf.drop(columns=[annual_source_order, cf_report_type, cf_end_date])

    # 对于固定资产年报面板：合计行 + Q4；无公告日字段，用报告期 + 4 个月代理
    fas = fas_df[fas_required].copy()
    fas[annual_source_order] = np.arange(len(fas))
    fas = fas.loc[fas[fasdepr_type].eq(fasdepr_type_total)]
    fas = fas.loc[fas[fas_report_type].eq('Q4')]
    fas = fas.sort_values([fas_code, fas_end_date, annual_source_order])
    fas = fas.drop_duplicates([fas_code, fas_end_date], keep='last')
    fas = fas.rename(columns={fas_code: annual_stock_code})
    fas['report_year'] = fas[fas_end_date].dt.year
    fas['fas_pub'] = fas[fas_end_date] + pd.DateOffset(months=4)
    fas = fas.drop(columns=[
        annual_source_order, fas_report_type, fas_end_date, fasdepr_type,
    ])

    # 三表在 (股票, 报告年) 上合成宽表
    annual = bs.merge(cf, on=[annual_stock_code, 'report_year'], how='outer')
    annual = annual.merge(fas, on=[annual_stock_code, 'report_year'], how='outer')
    annual = annual.sort_values([annual_stock_code, 'report_year'])

    grouped = annual.groupby(annual_stock_code)
    previous_end = grouped[bs_end_date].shift(1)
    adjacent = (annual['report_year'] - previous_end.dt.year).eq(1)

    delta_cur_assets = (
        annual[total_cur_assets] - grouped[total_cur_assets].shift(1)
    ).where(adjacent)
    delta_cur_lia = (
        annual[total_cur_lia] - grouped[total_cur_lia].shift(1)
    ).where(adjacent)
    delta_tax_pay = (
        annual[tax_pay] - grouped[tax_pay].shift(1)
    ).where(adjacent)
    # 短期借款缺失按「未列示≈无借款」口径填 0；整行没有资产负债表时保持缺失。
    short_loan_level = (
        annual[short_loan].fillna(0).where(annual[bs_end_date].notna())
    )
    delta_short_loan = (
        short_loan_level
        - short_loan_level.groupby(annual[annual_stock_code]).shift(1)
    ).where(adjacent)

    # Dep = 固定资产折旧 + 无形资产摊销 + 长期待摊费用摊销（摊销列内缺失当 0）。
    depreciation = (
        annual[fas_depr_incr]
        + annual[cf_intangible_amort].fillna(0)
        + annual[cf_defer_amort].fillna(0)
    )
    annual[annual_value] = (
        (delta_cur_assets - annual[cf_net_increase])
        - (delta_cur_lia - delta_short_loan - delta_tax_pay)
        - depreciation
    )
    # 可用日 = 三类年报公告日里最晚的一个（更正披露也覆盖）。
    annual[annual_available_date] = (
        annual[['bs_pub', 'cf_pub', 'fas_pub']].max(axis=1)
    )
    annual = annual.loc[
        np.isfinite(annual[annual_value])
        & annual[annual_available_date].notna()
    ]
    # 同一可用日出现多个报告期时（如招股书同批披露），只保留较新的报告期。
    annual = annual.sort_values([
        annual_stock_code, annual_available_date, 'report_year',
    ])
    annual = annual.drop_duplicates(
        [annual_stock_code, annual_available_date], keep='last'
    )
    return annual[[
        annual_stock_code, annual_available_date, annual_value,
    ]].sort_values([
        annual_available_date, annual_stock_code,
    ]).reset_index(drop=True)


def prepare_annual_noa(bs_df):
    """折叠资产负债表为「年度净经营资产」事件表。
    NOA_t = (OA_t − OL_t) / TA_{t−1}，OA = 总资产 − 货币资金 − 交易性金融资产，
    OL = 总负债 − 短期借款 − 长期借款；借款与交易性金融资产缺失按「未列示≈无此项」填 0。
    """
    required = [
        bs_stock_code, bs_end_date, bs_public_date, total_ass,
        cash_eqv, trad_ass, total_liab, short_loan, long_loan,
    ]
    require_columns(bs_df, required, '资产负债表')

    annual = bs_df[required].copy()
    annual[annual_source_order] = np.arange(len(annual))
    annual = annual.loc[annual[bs_end_date].dt.month.eq(12)]
    annual = annual.sort_values([
        bs_stock_code, bs_end_date, bs_public_date, annual_source_order,
    ])
    annual = annual.drop_duplicates([bs_stock_code, bs_end_date], keep='last')
    annual = annual.rename(columns={bs_stock_code: annual_stock_code})
    annual['report_year'] = annual[bs_end_date].dt.year
    annual = annual.sort_values([annual_stock_code, 'report_year'])

    operating_assets = (
        annual[total_ass] - annual[cash_eqv] - annual[trad_ass].fillna(0)
    )
    operating_liab = (
        annual[total_liab]
        - annual[short_loan].fillna(0)
        - annual[long_loan].fillna(0)
    )

    grouped = annual.groupby(annual_stock_code)
    previous_assets = grouped[total_ass].shift(1)
    previous_end = grouped[bs_end_date].shift(1)
    previous_pub = grouped[bs_public_date].shift(1)
    adjacent = (annual['report_year'] - previous_end.dt.year).eq(1)

    annual[annual_value] = (
        (operating_assets - operating_liab)
        / previous_assets.where(previous_assets > 0)
    ).where(adjacent)
    # 可用日 = max(本年年报公告日, 上一年年报公告日)——两份报表都齐才能算 NOA。
    annual['previous_pub'] = previous_pub
    annual[annual_available_date] = (
        annual[[bs_public_date, 'previous_pub']].max(axis=1)
    )
    annual = annual.loc[
        np.isfinite(annual[annual_value])
        & annual[annual_available_date].notna()
    ]
    # 同一可用日出现多个报告期时（如招股书同批披露），只保留较新的报告期。
    annual = annual.sort_values([
        annual_stock_code, annual_available_date, 'report_year',
    ])
    annual = annual.drop_duplicates(
        [annual_stock_code, annual_available_date], keep='last'
    )
    return annual[[
        annual_stock_code, annual_available_date, annual_value,
    ]].sort_values([
        annual_available_date, annual_stock_code,
    ]).reset_index(drop=True)